In [ ]:
import pyActigraphy
import plotly.graph_objects as go
import os
import pandas as pd
import numpy as np
import os

In [ ]:
#PROXIMOS PASSOS: juntar todos os files, calcular daily mean and median, daily M10 and L5, daily TAT 250 (verificar quanto da pra lux e nao melanopic) e daily 2000 lux, tambem
# calcular TAT based on sleep periods (after waking up until up to 4 hours before bedtime as daylight; after that, evening light; after sleep, light at night)
# waking up: sleep detection/alarm clock 
# calculate summary statistics using 12h bins

In [ ]:
#fpath = 'C:\\Users\\dc00955\\OneDrive - University of Surrey\\Desktop\\Sara_case_report\\'
fpath = 'C:\\Users\\gg00642\\OneDrive - University of Surrey\\Desktop\\Actigraphy Sara'

In [ ]:
# first joined file (from 21/09/2022 to 25/08/2024)
#raw = pyActigraphy.io.read_raw_mtn(fpath+'Joined_24-09-23_22-09-21.mtn')

# second joined file (from 23/09/2024 to 03/12/2024 )
#raw = pyActigraphy.io.read_raw_mtn(fpath+'2.0_Joined_23-09-24_03-12-24.mtn')

# third joined file (from 03/12/2024 to 31/01/2025)
#raw = pyActigraphy.io.read_raw_mtn(fpath+'Joined_03-12-24_03-02-25.mtn')

raw = pyActigraphy.io.read_raw_mtn(fpath+'\\1.0_Joined_23-09-24_03-02-25.mtn')

In [ ]:
#raw = pyActigraphy.io.read_raw_mtn(fpath+'Joined_03-12-24_03-02-25.mtn',
#                                   start_time = "2024-12-03 13:00:00",
#                                   period = "59 days") # dec-jan 2025

In [ ]:
layout = go.Layout(
    title="Actigraphy data",
    xaxis=dict(title="Date time"),
    yaxis=dict(title="Counts/period"),
    showlegend=False
)
go.Figure(data=[go.Scatter(x=raw.data.index.astype(str), y=raw.data)], layout=layout)

In [ ]:
raw.duration()

In [ ]:
raw.light

In [ ]:
raw.light.get_channel_list()

In [ ]:
layout = go.Layout(
    title="Light exposure data",
    xaxis=dict(title="Date time"),
    yaxis=dict(title=" Lux(log scale)"),
    showlegend=False
)
fig1 = go.Figure(
    data=[go.Scatter(
        x=raw.light.get_channel('whitelight').index.astype(str),
        y=raw.light.get_channel('whitelight'),
        name='White light')
    ],
    layout=layout
)
fig1.show()

In [ ]:
# APPLY MASKING BEFORE CALCULATING METRICS

In [ ]:
#raw = pyActigraphy.io.read_raw_mtn(fpath+'Joined_24-09-23_22-09-21.mtn')

In [ ]:
raw.light.create_light_mask()

In [ ]:
# manually adding mask to more than one period (FOR FIRST JOINED FILED (SEPT 2022 TO AUGUST 2024))
periods = [{'start': '2022-12-28 12:00:00', 'stop': '2022-12-29 12:00:00'},{'start': '2023-03-04 22:40:00', 'stop': '2023-04-18 10:00:00'},{'start': '2023-05-14 22:30:00', 'stop': '2023-05-15 07:15:00'},
           {'start': '2023-05-16 09:00:00', 'stop': '2023-06-29 13:00:00'},{'start': '2023-07-31 19:15:00', 'stop': '2023-08-16 12:00:00'},{'start': '2023-08-18 10:00:00', 'stop': '2023-08-22 13:00:00'},
           {'start': '2024-06-27 10:52:00', 'stop': '2024-06-28 10:40:00'}, {'start': '2024-07-30 19:15:00', 'stop': '2024-08-04 19:15:00'}]
for period in periods:
    raw.light.add_light_mask_period(start=period['start'], stop=period['stop'], channel = "whitelight")

In [ ]:
# manually adding mask to more than one period
periods = [{'start': '2023-03-04 22:00:00', 'stop': '2023-04-18 09:15:00'},{'start': '2023-05-16 06:30:00', 'stop': '2023-06-29 16:00:00'},{'start': '2023-07-28 14:00:00', 'stop': '2023-08-22 16:00:00'}]
for period in periods:
    raw.light.add_light_mask_period(start=period['start'], stop=period['stop'], channel = "whitelight")

In [ ]:
# manually adding mask to more than one period (THIRD JOINED FILE)
periods = [{'start': '2024-12-18 22:00:00', 'stop': '2024-12-19 07:10:00'},{'start': '2025-01-02 13:00:00', 'stop': '2025-01-03 09:00:00'},{'start': '2025-01-31 17:00:00', 'stop': '2025-02-03 13:15:00'}]
for period in periods:
    raw.light.add_light_mask_period(start=period['start'], stop=period['stop'], channel = "whitelight")

In [ ]:
raw.light.apply_mask = True

In [ ]:
raw.light.apply_mask 

In [ ]:
# visualising masked periods
layout = go.Layout(
    xaxis=dict(title="Date time"),
    yaxis=dict(title="Activity counts/period"),
    yaxis2=dict(title='Mask',overlaying='y',side='right'),
    showlegend=True
)
fig2 = go.Figure([
    go.Scatter(
        x=raw.light.get_channel('whitelight').index.astype(str),
        y=raw.light.get_channel('whitelight'),
        name='Light'),
    go.Scatter(
        x=raw.light.mask.index.astype(str),
        y=raw.light.mask.loc[:,'whitelight'],
        yaxis='y2', opacity=0.5,
        name='Mask')
], layout=layout)
fig2.show()

In [ ]:
# converting 250 melanopic lux to photopic lux (not that simple) - photopic lux = melanopic EDI/melanopic ratio
# standard office lighting (3500 K LED) with 300 lux at the task plane and 150 lux at the eye has a melanopic ratio (melanopic DER) of about 0.51 (which I used here to convert)
# REF: https://pmc.ncbi.nlm.nih.gov/articles/PMC8215265/
# also here: https://www.fagerhult.com/knowledge/light-and-people/melanopic-ratio/a-new-way-to-report-the-biological-impact-of-lighting/#:~:text=Melanopic%20Ratio%20(Melanopic%20Daylight%20Efficacy,the%20reference%20for%20Melanopic%20Ratio.
# https://journals.sagepub.com/doi/epub/10.1177/14771535221120350
# https://issuu.com/designinglighting/docs/dec_2022/s/18078622

# conversion: photopic lux = 250 melanopic EDI / 0.51 (melanopic DER) = 490 lux
# converted to log scale (to use it in the TAT treshold): np.log10(490+1) = 2.69

# treshold for outdoor light exposure = 2500 lux
# converted to log scale: 3.398

In [ ]:
np.log10(490+1)

In [ ]:
np.log10(2500+1)

In [ ]:
raw.light.light_exposure_level(agg='median')

In [ ]:
raw.light.summary_statistics_per_time_bin()

In [ ]:
light_values = raw.light.summary_statistics_per_time_bin()

In [ ]:
# Convert light values into a pandas DataFrame
light_summary = pd.DataFrame(light_values)
# Save to a CSV file
light_summary.to_csv("daily_light_summary_part3.csv", index=False)


In [ ]:
a=raw.light.summary_statistics_per_time_bin(
    bins=[
        ['2024-09-23 07:00:00','2024-09-23 19:00:00'],
        ['2024-09-23 19:00:00','2024-09-24 07:00:00'],
        ['2024-09-24 07:00:00','2024-09-24 19:00:00'],
        ['2024-09-24 19:00:00','2024-09-25 07:00:00'],
        ['2024-09-25 07:00:00','2024-09-25 19:00:00'],
        ['2024-09-25 19:00:00','2024-09-26 07:00:00'],
        ['2024-09-26 07:00:00','2024-09-26 19:00:00'],
        ['2024-09-26 19:00:00','2024-09-27 07:00:00'],
        ['2024-09-27 07:00:00','2024-09-27 19:00:00'],
        ['2024-09-27 19:00:00','2024-09-28 07:00:00'],
        ['2024-09-28 07:00:00','2024-09-28 19:00:00'],
        ['2024-09-28 19:00:00','2024-09-29 07:00:00'],
        ['2024-09-29 07:00:00','2024-09-29 19:00:00'],
        ['2024-09-29 19:00:00','2024-09-30 07:00:00'],
        ['2024-09-30 07:00:00','2024-09-30 19:00:00'],
        ['2024-09-30 19:00:00','2024-10-01 07:00:00'],
        ['2024-10-01 07:00:00','2024-10-01 19:00:00'],
        ['2024-10-01 19:00:00','2024-10-02 07:00:00'],
        ['2024-10-02 07:00:00','2024-10-02 19:00:00'],
        ['2024-10-02 19:00:00','2024-10-03 07:00:00'],
        ['2024-10-03 07:00:00','2024-10-03 19:00:00'],
        ['2024-10-03 19:00:00','2024-10-04 07:00:00'],
        ['2024-10-04 07:00:00','2024-10-04 19:00:00'],
        ['2024-10-04 19:00:00','2024-10-05 07:00:00'],
        ['2024-10-05 07:00:00','2024-10-05 19:00:00'],
        ['2024-10-05 19:00:00','2024-10-06 07:00:00'],
        ['2024-10-06 07:00:00','2024-10-06 19:00:00'],
        ['2024-10-06 19:00:00','2024-10-07 07:00:00'],
        ['2024-10-07 07:00:00','2024-10-07 19:00:00'],
        ['2024-10-07 19:00:00','2024-10-08 07:00:00'],
        ['2024-10-08 07:00:00','2024-10-08 19:00:00'],
        ['2024-10-08 19:00:00','2024-10-09 07:00:00'],
        ['2024-10-09 07:00:00','2024-10-09 19:00:00'],
        ['2024-10-09 19:00:00','2024-10-10 07:00:00'],
        ['2024-10-10 07:00:00','2024-10-10 19:00:00'],
        ['2024-10-10 19:00:00','2024-10-11 07:00:00'],
        ['2024-10-11 07:00:00','2024-10-11 19:00:00'],
        ['2024-10-11 19:00:00','2024-10-12 07:00:00'],
        ['2024-10-12 07:00:00','2024-10-12 19:00:00'],
        ['2024-10-12 19:00:00','2024-10-13 07:00:00'],
        ['2024-10-13 07:00:00','2024-10-13 19:00:00'],
        ['2024-10-13 19:00:00','2024-10-14 07:00:00'],
        ['2024-10-14 07:00:00','2024-10-14 19:00:00'],
        ['2024-10-14 19:00:00','2024-10-15 07:00:00'],
        ['2024-10-15 07:00:00','2024-10-15 19:00:00'],
        ['2024-10-15 19:00:00','2024-10-16 07:00:00'],
        ['2024-10-16 07:00:00','2024-10-16 19:00:00'],
        ['2024-10-16 19:00:00','2024-10-17 07:00:00'],
        ['2024-10-17 07:00:00','2024-10-17 19:00:00'],
        ['2024-10-17 19:00:00','2024-10-18 07:00:00'],
        ['2024-10-18 07:00:00','2024-10-18 19:00:00'],
        ['2024-10-18 19:00:00','2024-10-19 07:00:00'],
        ['2024-10-19 07:00:00','2024-10-19 19:00:00'],
        ['2024-10-19 19:00:00','2024-10-20 07:00:00'],
        ['2024-10-20 07:00:00','2024-10-20 19:00:00'],
        ['2024-10-20 19:00:00','2024-10-21 07:00:00'],
        ['2024-10-21 07:00:00','2024-10-21 19:00:00'],
        ['2024-10-21 19:00:00','2024-10-22 07:00:00'],
        ['2024-10-22 07:00:00','2024-10-22 19:00:00'],
        ['2024-10-22 19:00:00','2024-10-23 07:00:00'],
        ['2024-10-23 07:00:00','2024-10-23 19:00:00'],
        ['2024-10-23 19:00:00','2024-10-24 07:00:00'],
        ['2024-10-24 07:00:00','2024-10-24 19:00:00'],
        ['2024-10-24 19:00:00','2024-10-25 07:00:00'],
        ['2024-10-25 07:00:00','2024-10-25 19:00:00'],
        ['2024-10-25 19:00:00','2024-10-26 07:00:00'],
        ['2024-10-26 07:00:00','2024-10-26 19:00:00'],
        ['2024-10-26 19:00:00','2024-10-27 07:00:00'],
        ['2024-10-27 07:00:00','2024-10-27 19:00:00'],
        ['2024-10-27 19:00:00','2024-10-28 07:00:00'],
        ['2024-10-28 07:00:00','2024-10-28 19:00:00'],
        ['2024-10-28 19:00:00','2024-10-29 07:00:00'],
        ['2024-10-29 07:00:00','2024-10-29 19:00:00'],
        ['2024-10-29 19:00:00','2024-10-30 07:00:00'],
        ['2024-10-30 07:00:00','2024-10-30 19:00:00'],
        ['2024-10-30 19:00:00','2024-10-31 07:00:00'],
        ['2024-10-31 07:00:00','2024-10-31 19:00:00'],
        ['2024-10-31 19:00:00','2024-11-01 07:00:00'],
        ['2024-11-01 07:00:00','2024-11-01 19:00:00'],
        ['2024-11-01 19:00:00','2024-11-02 07:00:00'],
        ['2024-11-02 07:00:00','2024-11-02 19:00:00'],
        ['2024-11-02 19:00:00','2024-11-03 07:00:00'],
        ['2024-11-03 07:00:00','2024-11-03 19:00:00'],
        ['2024-11-03 19:00:00','2024-11-04 07:00:00'],
        ['2024-11-04 07:00:00','2024-11-04 19:00:00'],
        ['2024-11-04 19:00:00','2024-11-05 07:00:00'],
        ['2024-11-05 07:00:00','2024-11-05 19:00:00'],
        ['2024-11-05 19:00:00','2024-11-06 07:00:00'],
        ['2024-11-06 07:00:00','2024-11-06 19:00:00'],
        ['2024-11-06 19:00:00','2024-11-07 07:00:00'],
        ['2024-11-07 07:00:00','2024-11-07 19:00:00'],
        ['2024-11-07 19:00:00','2024-11-08 07:00:00'],
        ['2024-11-08 07:00:00','2024-11-08 19:00:00'],
        ['2024-11-08 19:00:00','2024-11-09 07:00:00'],
        ['2024-11-09 07:00:00','2024-11-09 19:00:00'],
        ['2024-11-09 19:00:00','2024-11-10 07:00:00'],
        ['2024-11-10 07:00:00','2024-11-10 19:00:00'],
        ['2024-11-10 19:00:00','2024-11-11 07:00:00'],
        ['2024-11-11 07:00:00','2024-11-11 19:00:00'],
        ['2024-11-11 19:00:00','2024-11-12 07:00:00'],
        ['2024-11-12 07:00:00','2024-11-12 19:00:00'],
        ['2024-11-12 19:00:00','2024-11-13 07:00:00'],
        ['2024-11-13 07:00:00','2024-11-13 19:00:00'],
        ['2024-11-13 19:00:00','2024-11-14 07:00:00'],
        ['2024-11-14 07:00:00','2024-11-14 19:00:00'],
        ['2024-11-14 19:00:00','2024-11-15 07:00:00']
    ], 
    agg_func=['mean']
 )


In [ ]:
# add the date column to the dataframe "a" 
a['date'] = ['2024-09-23', '2024-09-23/24', '2024-09-24', '2024-09-24/25', '2024-09-25', '2024-09-25/26', '2024-09-26', '2024-09-26/27', '2024-09-27', '2024-09-27/28', '2024-09-28', '2024-09-28/29', '2024-09-29', '2024-09-29/30', '2024-09-30', '2024-09-30/10-01', '2024-10-01', '2024-10-01/02', '2024-10-02', '2024-10-02/03', '2024-10-03', '2024-10-03/04', '2024-10-04', '2024-10-04/05', '2024-10-05', '2024-10-05/06', '2024-10-06', '2024-10-06/07', '2024-10-07', '2024-10-07/08', '2024-10-08', '2024-10-08/09', '2024-10-09', '2024-10-09/10', '2024-10-10', '2024-10-10/11', '2024-10-11', '2024-10-11/12', '2024-10-12', '2024-10-12/13', '2024-10-13', '2024-10-13/14', '2024-10-14', '2024-10-14/15', '2024-10-15', '2024-10-15/16', '2024-10-16', '2024-10-16/17', '2024-10-17', '2024-10-17/18', '2024-10-18', '2024-10-18/19', '2024-10-19', '2024-10-19/20', '2024-10-20', '2024-10-20/21', '2024-10-21', '2024-10-21/22', '2024-10-22', '2024-10-22/23', 
            '2024-10-23', '2024-10-23/24', '2024-10-24', '2024-10-24/25', '2024-10-25', '2024-10-25/26', '2024-10-26', '2024-10-26/27', '2024-10-27', '2024-10-27/28', '2024-10-28', '2024-10-28/29', '2024-10-29', '2024-10-29/30', '2024-10-30', '2024-10-30/31', '2024-10-31', '2024-10-31/11-01', '2024-11-01', '2024-11-01/02', '2024-11-02', '2024-11-02/03', '2024-11-03', '2024-11-03/04', '2024-11-04', '2024-11-04/05', '2024-11-05', '2024-11-05/06', '2024-11-06', '2024-11-06/07', '2024-11-07', '2024-11-07/08', '2024-11-08', '2024-11-08/09', '2024-11-09', '2024-11-09/10', '2024-11-10', '2024-11-10/11', '2024-11-11', '2024-11-11/12', '2024-11-12', '2024-11-12/13', '2024-11-13', '2024-11-13/14', '2024-11-14', '2024-11-14/15']

In [ ]:
raw.light.light_exposure_level(
    start_time='07:00',
    stop_time='19:00',
    agg='mean'
)

In [ ]:
raw.light.light_exposure_level(
    threshold=1, # on a log10 scale
    start_time='07:00',
    stop_time='19:00'
)

In [ ]:
# daily TAT above 250 melanopic EDI (~490 lux) (and above 2500 lux)
daily_TAT = raw.light.TATp(threshold= 3.398, oformat='minute') # 3.398 to 2500 lux; 2.69 to 490 lux

In [ ]:
# Convert daily TAT into a pandas DataFrame
light_TAT = pd.DataFrame(daily_TAT)
# Save to a CSV file
light_TAT.to_csv("daily_TAT_2500_part3.csv", index=False)


In [ ]:
daily_TAT

In [ ]:
raw.light.LMX(length='10h',lowest=False)

In [ ]:
# Check if start_time and end_time are None
if raw.light.start_time is None or raw.light.stop_time is None:
    # Manually set start_time and end_time based on the data index
    start_time = raw.light.data.index[0]  # First timestamp in the data
    end_time = raw.light.data.index[-1]   # Last timestamp in the data
else:
    start_time = raw.light.start_time
    end_time = raw.light.end_time

In [ ]:
# Initialize lists to store daily M10 and L5 values
daily_m10 = []
daily_l5 = []
daily_m10_start = []
daily_l5_start = []
dates = []

# Get the start and end times of the recording
start_time = raw.light.data.index[0]  # First timestamp in the data
end_time = raw.light.data.index[-1]   # Last timestamp in the data

# Iterate over each day in the dataset
current_time = start_time
while current_time < end_time:
    # Define the start and end of the current day
    day_start = current_time
    day_end = day_start + pd.Timedelta(days=1)
    
    # Extract the light data for the current day using pandas filtering
    day_data = raw.light.data.loc[day_start:day_end]
    
    # Check if there is data for the current day
    if len(day_data) > 0:
        # Calculate M10 for the current day
        m10_series = day_data.rolling(window='10h').mean().max()  # M10 level (as a Series)
        m10 = m10_series.iloc[0]  # Extract the raw value from the Series
        m10_start_series = day_data.rolling(window='10h').mean().idxmax()  # M10 start time (as a Series)
        m10_start = m10_start_series.iloc[0]  # Extract the raw timestamp from the Series
        
        # Calculate L5 for the current day
        l5_series = day_data.rolling(window='5h').mean().min()  # L5 level (as a Series)
        l5 = l5_series.iloc[0]  # Extract the raw value from the Series
        l5_start_series = day_data.rolling(window='5h').mean().idxmin()  # L5 start time (as a Series)
        l5_start = l5_start_series.iloc[0]  # Extract the raw timestamp from the Series
        
        # Append results to lists
        daily_m10.append(m10)
        daily_m10_start.append(m10_start)
        daily_l5.append(l5)
        daily_l5_start.append(l5_start)
        dates.append(day_start.date())
    else:
        # If no data is available for the day, append NaN or None
        daily_m10.append(None)
        daily_m10_start.append(None)
        daily_l5.append(None)
        daily_l5_start.append(None)
        dates.append(day_start.date())
    
    # Move to the next day
    current_time = day_end

# Create a DataFrame to store the daily results
daily_results = pd.DataFrame({
    'Date': dates,  # List of dates
    'M10_Level': daily_m10,  # M10 level for each day (raw value)
    'M10_Start': daily_m10_start,  # M10 start time for each day (raw timestamp)
    'L5_Level': daily_l5,  # L5 level for each day (raw value)
    'L5_Start': daily_l5_start  # L5 start time for each day (raw timestamp)
})

# Save the results to a CSV file
daily_results.to_csv('daily_m10_l5_results.csv', index=False)

# Print the results
print(daily_results)